# Ilera-ATLAS · 03_translate: N-ATLaS translates reviewed items into Hausa and Yoruba

**Runbook Part 5.** Run it **twice**:

| Run | `STAGE` | When | What | Review |
|---|---|---|---|---|
| 1 | `"bench"` | **Sun 4 Oct** | 150 Hausa + 150 Yoruba benchmark items | native speaker checks **every** row before the Mon 5 Oct freeze |
| 2 | `"train"` | Mon 5 Oct (Commit mode) | 1,000 Hausa + 1,000 Yoruba training items | all flagged rows + 1 in 5 of the rest, by Tue 6 Oct |

**Input:** the English sheet after Part 4 review. In Google Sheets: File → Download → CSV for each tab,
combine them into **`review_en_final.csv`** (same columns as the drafts), then upload it to your Hugging
Face dataset under `reviewed/` (dataset page → Files → Add file → Upload files).

**How translation works (safer than translating whole answers):** the notebook translates the
*question*, the *WHAT TO DO steps* and the *DANGER SIGNS* separately, then rebuilds the answer itself. So
`ACTION:` (TREAT / REFER / URGENT), the four labels and the `SOURCE:` line stay in exact English and the
same parser works in every language.

**Automatic flags** tell reviewers where to look first: changed numbers/doses, wrong language
(GlotLID), suspicious length, repetition loops, untranslated text.

**Kaggle:** GPU T4 x2 · Internet On · secret `HF_TOKEN`. Smoke test first with `LIMIT = 6`.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"          # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"
STAGE = "bench"                        # "bench" (Sun) or "train" (Mon)
N_PER_LANG = {"bench": 150, "train": 1000}
LANGS = ["ha", "yo"]                   # add "ig" if you have an Igbo reviewer
LIMIT = 6                              # smoke test: items per language. Set None for the full run.
BATCH = 8                              # strings per generation batch (short texts); lower if OOM
MAX_NEW = 600
SAVE_EVERY = 25                        # batches between progress uploads
UPLOAD = True
EN_CSV = "reviewed/review_en_final.csv"   # path inside the HF dataset
LOCAL_EN_CSV = None                    # or a local path (offline testing / Kaggle input file)
USE_LID = True                         # GlotLID language check (downloads a 1.6 GB model once)
SMOKE_TEST_UNREVIEWED = False          # True = test the translation machinery on UNREVIEWED drafts
                                       #        (needs LIMIT; nothing is uploaded; output is NOT usable data)

In [ ]:
import os
import subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated Kaggle default -> removes the FutureWarning
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # its replacement: faster model downloads
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate"], capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")
# Language check (GlotLID) needs fastText. "fasttext-predict" has ready-made wheels for Kaggle's
# Python 3.13 and no NumPy dependency; "fasttext-wheel" stops at 3.12. Optional: if it fails, the
# notebook still runs and only the language check is skipped.
r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "fasttext-predict"],
                    capture_output=True, text=True)
print("fasttext-predict installed" if r2.returncode == 0 else "fasttext unavailable: language check will be skipped")
# If the next cells fail with an import error: Run > Restart kernel, then Run All again.

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    import glob                                     # API runs: token in a private Kaggle dataset
    for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True):
        return open(p).read().strip()
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_EN_CSV:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN
if SMOKE_TEST_UNREVIEWED:
    assert LIMIT, "SMOKE_TEST_UNREVIEWED needs LIMIT (e.g. LIMIT = 6)"
    UPLOAD = False
    if not LOCAL_EN_CSV:
        EN_CSV = "drafts/review_en_vignette.csv"     # the unreviewed draft sheet from 02_draft
    print("*** SMOKE TEST ON UNREVIEWED DRAFTS: status and key_facts are ignored, nothing is uploaded. "
          "Use it only to check that translation works; the output is NOT usable data. ***")

## 0. Upload the reviewed English sheet (once, after Part 4)
1. In Google Sheets, download each reviewed tab as CSV (File → Download → .csv). Usually two files:
   the vignette sheet and the qa sheet.
2. In this Kaggle notebook: right panel → **Input → Upload → New Dataset**, add the CSV file(s), name it
   e.g. `ilera-reviewed`, Create. The files appear under `/kaggle/input/...`.
3. Run the cell: with `REVIEWED_FILES = "auto"` it finds every attached CSV that has the review-sheet columns (or list the paths yourself). It checks the columns, merges the files,
   and uploads them as `reviewed/review_en_final.csv` to your Hugging Face dataset (the file this
   notebook and 03b_build read). Re-run it whenever the reviewers send a newer export.

In [ ]:
import glob, collections
REVIEWED_FILES = "auto"    # "auto" = every review-sheet CSV/XLSX attached under /kaggle/input; or a list of paths; [] = skip
import csv as _csv
need = ["id", "split", "kind", "chunk_id", "source", "question", "action", "answer",
        "key_facts", "status", "reviewer", "comment"]

def _is_review_sheet(path):
    try:
        with open(path, encoding="utf-8-sig") as fh:
            return set(need[:10]) <= set(next(_csv.reader(fh)))
    except Exception:
        return False

def _xlsx_to_csv():
    # reviewers often save from Excel: convert every attached .xlsx sheet to a .csv first
    import pandas as pd
    out = []
    for x in sorted(glob.glob("/kaggle/input/**/*.xlsx", recursive=True)):
        os.makedirs("converted", exist_ok=True)
        for sheet, df in pd.read_excel(x, sheet_name=None, dtype=str).items():
            p = os.path.join("converted", f"{os.path.basename(x).split('.')[0]}__{sheet}.csv")
            df.fillna("").to_csv(p, index=False, encoding="utf-8-sig")
            out.append(p)
    return out

if REVIEWED_FILES == "auto":
    found = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)) + _xlsx_to_csv()
    REVIEWED_FILES = [p for p in found if _is_review_sheet(p)]
    print(f"auto: {len(found)} CSV file(s) under /kaggle/input, {len(REVIEWED_FILES)} look like review sheets:")
    for p in REVIEWED_FILES:
        print("   ", p)
def _header(path):
    with open(path, encoding="utf-8-sig") as fh:
        return next(_csv.reader(fh))

# translated (Hausa/Yoruba) review sheets have en_id + lang columns: they go to their own file
HY_FILES = [f for f in REVIEWED_FILES if {"en_id", "lang"} <= set(_header(f))]
EN_FILES = [f for f in REVIEWED_FILES if f not in HY_FILES]
if HY_FILES:
    from huggingface_hub import HfApi
    hy_rows, hy_seen, hy_cols = [], set(), []
    for f in HY_FILES:
        for col in _header(f):
            if col not in hy_cols:
                hy_cols.append(col)
        for r in _csv.DictReader(open(f, encoding="utf-8-sig")):
            if r["id"] and r["id"] not in hy_seen:
                hy_seen.add(r["id"]); hy_rows.append(r)
    with open("review_ha_yo_final.csv", "w", newline="", encoding="utf-8-sig") as out:
        w = _csv.DictWriter(out, fieldnames=hy_cols, extrasaction="ignore"); w.writeheader(); w.writerows(hy_rows)
    hy_ok = collections.Counter(r["lang"] for r in hy_rows if r["status"].strip().lower() in ("ok", "fixed"))
    print(f"Hausa/Yoruba: {len(hy_rows)} rows from {len(HY_FILES)} file(s); marked ok/fixed per language: {dict(hy_ok)}")
    if not SMOKE_TEST_UNREVIEWED:
        assert hy_ok, "No translated row has status ok/fixed: native speakers must review them first."
        HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="review_ha_yo_final.csv",
                                          path_in_repo="reviewed/review_ha_yo_final.csv", repo_id=DATA_REPO,
                                          repo_type="dataset", commit_message="reviewed Hausa/Yoruba sheet")
        print(f"uploaded -> {DATA_REPO}/reviewed/review_ha_yo_final.csv")
REVIEWED_FILES = EN_FILES
if REVIEWED_FILES:
    from huggingface_hub import HfApi
    merged, seen = [], set()
    for f in REVIEWED_FILES:
        rows_f = list(_csv.DictReader(open(f, encoding="utf-8-sig")))
        miss = set(need[:10]) - set(rows_f[0])
        assert not miss, f"{f} is missing columns {miss}: export the review sheet with all its columns"
        for r in rows_f:
            if r["id"] and r["id"] not in seen:
                seen.add(r["id"]); merged.append({k: r.get(k, "") for k in need})
    with open("review_en_final.csv", "w", newline="", encoding="utf-8-sig") as out:
        w = _csv.DictWriter(out, fieldnames=need); w.writeheader(); w.writerows(merged)
    ok = sum(r["status"].strip().lower() in ("ok", "fixed") for r in merged)
    print(f"{len(merged)} rows merged from {len(REVIEWED_FILES)} file(s); {ok} marked ok/fixed")
    if len(merged) < 300 and not SMOKE_TEST_UNREVIEWED:
        print(f"NOTE: only {len(merged)} rows. A full 02_draft run gives ~1,500 vignette rows; "
              "this looks like smoke-test output (LIMIT = 4).")
    if SMOKE_TEST_UNREVIEWED:
        LOCAL_EN_CSV = "review_en_final.csv"          # use the attached file locally, upload nothing
        print("smoke test: using the attached sheet locally, not uploading it")
    assert ok or SMOKE_TEST_UNREVIEWED, (
        "No row has status ok/fixed: these look like unreviewed drafts. Reviewers must set the status "
        "column in Google Sheets (Part 4) before exporting. To only test the translation machinery now, "
        "set SMOKE_TEST_UNREVIEWED = True and LIMIT = 6 in Config.")
if REVIEWED_FILES and not SMOKE_TEST_UNREVIEWED:
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="review_en_final.csv", path_in_repo=EN_CSV,
                                      repo_id=DATA_REPO, repo_type="dataset",
                                      commit_message="reviewed English sheet")
    print(f"uploaded -> {DATA_REPO}/{EN_CSV}")
elif not REVIEWED_FILES and not HY_FILES:
    print("No review sheet to upload: skipping (fine if reviewed/review_en_final.csv is already on Hugging Face). "
          "To upload: attach the exported CSVs via Input > Upload, then re-run this cell.")

## 0a. Pre-fill key-fact suggestions for the reviewers (optional, before Part 4)
Filling `key_facts` is the slowest part of the review. This cell reads the **draft** sheet from
`02_draft` (`drafts/review_en_vignette.csv` on Hugging Face), suggests 2–4 key facts for every
benchmark and probe row, and writes `review_en_vignette_suggested.csv` (Output tab, and
`drafts/` on Hugging Face). Import that file into Google Sheets instead of the plain draft.

Suggestions are taken **word-for-word from the row's own answer** (drug names, doses, durations,
first danger signs), so they can always be scored. They are **only suggestions**: the clinical
reviewer must confirm or edit them, and `status` stays empty until a human sets it. Rows that already
have `key_facts` are not touched. The `comment` column says "key facts auto-suggested: confirm".

In [ ]:
import re
SUGGEST_KEY_FACTS = False                      # True = run this helper
SUGGEST_FROM = "drafts/review_en_vignette.csv"  # draft sheet in the HF dataset (or a local path)

DRUGS = ["amoxicillin", "cotrimoxazole", "co-trimoxazole", "paracetamol", "ibuprofen", "ors",
         "oral rehydration", "zinc", "artemether", "lumefantrine", "artesunate", "act", "quinine",
         "metronidazole", "ciprofloxacin", "erythromycin", "tetracycline", "chloramphenicol",
         "gentamicin", "benzathine", "penicillin", "ceftriaxone", "ferrous", "iron", "folic acid",
         "vitamin a", "albendazole", "mebendazole", "praziquantel", "chlorhexidine", "gentian violet",
         "nystatin", "clotrimazole", "oxytocin", "misoprostol", "magnesium sulphate", "salbutamol",
         "hydrocortisone", "diazepam", "antacid", "sulfadoxine", "pyrimethamine", "sp", "tetanus toxoid",
         "td", "bcg", "opv", "pentavalent", "measles", "insecticide-treated net", "itn"]
DOSE = re.compile(r"\b\d+(?:[.,]\d+)?\s?(?:mg|mcg|g|ml|iu|units?|tabs?|tablets?|capsules?|drops?|"
                  r"days?|hours?|hrs?|weeks?|months?|times|kg|%|°c)\b", re.I)

def suggest_facts(answer, k=4):
    body = answer.split("SOURCE:")[0]
    low = body.lower()
    facts = []
    for d in DRUGS:                                   # drug names (whole word), in answer order
        m = re.search(rf"(?<![a-z]){re.escape(d)}(?![a-z])", low)
        if m:
            facts.append((m.start(), body[m.start():m.end()]))
    for m in DOSE.finditer(body):                     # doses and durations
        facts.append((m.start(), m.group(0)))
    danger = re.search(r"DANGER SIGNS:\s*(.+)", body)
    if danger and "none listed" not in danger.group(1).lower():
        for part in re.split(r"[,;]| and ", danger.group(1))[:2]:   # first two danger signs
            part = part.strip(" .")
            if 3 <= len(part) <= 40:
                facts.append((10**6 + body.find(part), part))
    out, seen = [], set()
    for _, f in sorted(facts):
        if f.lower() not in seen and f.lower() in low:
            seen.add(f.lower()); out.append(f)
    return out[:k]

assert suggest_facts("ACTION: REFER\nWHAT TO DO:\n1. Give amoxicillin 250 mg for 5 days.\n"
                     "DANGER SIGNS: chest indrawing, convulsion\nSOURCE: x") == \
       ["amoxicillin", "250 mg", "5 days", "chest indrawing"]

if SUGGEST_KEY_FACTS:
    import csv as _csv
    src = SUGGEST_FROM if os.path.exists(SUGGEST_FROM) else __import__("huggingface_hub").hf_hub_download(
        DATA_REPO, SUGGEST_FROM, repo_type="dataset", token=HF_TOKEN)
    rows_s = list(_csv.DictReader(open(src, encoding="utf-8-sig")))
    n_new, n_none = 0, 0
    for r in rows_s:
        if r["split"] in ("bench", "probe") and not r["key_facts"].strip():
            facts = suggest_facts(r["answer"])
            if facts:
                r["key_facts"] = "; ".join(facts); n_new += 1
                if not r.get("comment", "").strip():
                    r["comment"] = "key facts auto-suggested: confirm"
            else:
                n_none += 1
    out_name = os.path.basename(SUGGEST_FROM).replace(".csv", "_suggested.csv")
    with open(out_name, "w", newline="", encoding="utf-8-sig") as fh:
        w = _csv.DictWriter(fh, fieldnames=list(rows_s[0])); w.writeheader(); w.writerows(rows_s)
    print(f"suggested key facts for {n_new} bench/probe rows; {n_none} rows had nothing to suggest "
          f"(reviewer writes those) -> {out_name}")
    for r in [r for r in rows_s if r["comment"] == "key facts auto-suggested: confirm"][:3]:
        print("  ", r["id"], "|", r["key_facts"])
    if UPLOAD:
        from huggingface_hub import HfApi
        HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=out_name, path_in_repo=f"drafts/{out_name}",
                                          repo_id=DATA_REPO, repo_type="dataset",
                                          commit_message="key-fact suggestions for review")
        print(f"uploaded -> {DATA_REPO}/drafts/{out_name}")

## 1. Load the reviewed English items and pick what to translate
Only rows with `status` = `ok` or `fixed` are used. Benchmark rows must also have `key_facts`.
The Hausa and Yoruba benchmark sets are drawn from the **same** English benchmark items, so the three
languages form a parallel test: you can compare languages on identical questions.

In [ ]:
import csv, json, random, re, time, collections

path = LOCAL_EN_CSV
if not path:
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(DATA_REPO, EN_CSV, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        raise FileNotFoundError(
            f"{EN_CSV} is not in {DATA_REPO} yet. This notebook needs the English sheet AFTER the "
            "Part 4 review: export the reviewed Google Sheet as CSV (vignette + qa rows together, "
            "status = ok/fixed, key_facts filled for bench rows) and upload it as "
            f"{EN_CSV}. See the 'Upload the reviewed sheet' cell in this notebook. ({type(e).__name__})") from None
rows = list(csv.DictReader(open(path, encoding="utf-8-sig")))
need = {"id", "split", "kind", "chunk_id", "source", "question", "action", "answer", "key_facts", "status"}
missing = need - set(rows[0])
assert not missing, f"review_en_final.csv is missing columns: {missing}"
status_counts = collections.Counter(r["status"].strip() or "(empty)" for r in rows)
rows = [r for r in rows if SMOKE_TEST_UNREVIEWED or r["status"].strip().lower() in ("ok", "fixed")]
print(collections.Counter(r["split"] for r in rows), "reviewed English rows")

pool = [r for r in rows if r["split"] == STAGE]
if STAGE == "bench" and not SMOKE_TEST_UNREVIEWED:
    no_facts = [r["id"] for r in pool if not r["key_facts"].strip()]
    if no_facts:      # they could never be scored, and 03b_build excludes them anyway: skip, do not stop
        print(f"skipping {len(no_facts)} bench rows without key_facts (e.g. {no_facts[:3]})")
        pool = [r for r in pool if r["key_facts"].strip()]
n = N_PER_LANG[STAGE]
if STAGE == "train" and len(pool) < n:         # translate every reviewed training row if there are fewer
    print(f"only {len(pool)} reviewed train rows: translating all of them (target was {n})")
    n = len(pool)
if SMOKE_TEST_UNREVIEWED:
    assert pool, f"no {STAGE} rows in the sheet at all"
    n = min(n, len(pool))
if len(pool) < n:
    print("status values found in the sheet:", status_counts.most_common(8))
    print("Accepted: ok / fixed (drop = excluded). Empty status = not reviewed yet (runbook Part 4).")
assert len(pool) >= n, f"only {len(pool)} reviewed {STAGE} rows, need {n}"
picked = random.Random(1).sample(pool, n)          # same items for every language
if LIMIT:
    picked = picked[:LIMIT]
    print(f"*** SMOKE TEST: only {LIMIT} items per language. Set LIMIT = None for the full run. ***")
print(len(picked), "English items x", len(LANGS), "languages")

## 2. Split each answer into translatable parts
`ACTION` and `SOURCE` are kept from English. Only the steps and danger signs are translated.

In [ ]:
def split_answer(ans):
    action = re.search(r"ACTION:\s*(TREAT|REFER|URGENT)", ans).group(1)
    steps = re.search(r"WHAT TO DO:\s*\n?([\s\S]*?)(?=\n\s*DANGER SIGNS:|\n\s*SOURCE:|\Z)", ans)
    danger = re.search(r"DANGER SIGNS:\s*(.*)", ans)
    source = re.search(r"SOURCE:\s*(.*)", ans)
    return {"action": action,
            "steps": steps.group(1).strip() if steps else "",
            "danger": danger.group(1).strip() if danger else "none listed",
            "source": source.group(1).strip() if source else ""}

def build_answer(p):
    return (f"ACTION: {p['action']}\nWHAT TO DO:\n{p['steps']}\n"
            f"DANGER SIGNS: {p['danger']}\nSOURCE: {p['source']}")

_a = "ACTION: REFER\nWHAT TO DO:\n1. Give amoxicillin 250 mg.\n2. Refer today.\nDANGER SIGNS: convulsion\nSOURCE: CHEW Standing Orders 2024, Section 2.3 Fever, p. 50"
assert build_answer(split_answer(_a)) == _a
print("answer split/rebuild self-check passed")

## 3. Translation prompt
Rules given to N-ATLaS: keep numbers, doses, drug names and numbering; output the translation only.
Yoruba gets an extra instruction to write full tone marks (N-ATLaS's weakest language: 2.69/5 in its
own evaluation, so expect more fixes there).

In [ ]:
LANG_NAME = {"ha": "Hausa", "yo": "Yoruba", "ig": "Igbo"}
EXTRA = {"ha": "", "ig": "",
         "yo": "Write standard Yoruba with full tone marks and under-dots (for example: ọmọ, ìbà, ẹ̀jẹ̀). "}

T_PROMPT = """Translate the text below from English into {lang}.
Rules: keep all numbers, doses, units and drug names exactly as they are. Keep the numbered-list layout.
{extra}Write only the translation, nothing else.

Text:
{text}"""

def clean_output(s):
    s = s.strip()
    s = re.sub(r"^(Translation|Fassara|Ìtumọ̀|Itumo)\s*[:：]\s*", "", s, flags=re.I)
    return s.strip().strip('"').strip()

## 4. Load N-ATLaS (float16) and `chat()`

In [ ]:
import gc, sys, torch, transformers
from packaging import version
# Safe to re-run: free any model already loaded in this session (and the reference an earlier
# failed load keeps in its traceback) before loading again, otherwise the T4 runs out of memory.
for _name in ("trainer", "model"):
    globals().pop(_name, None)
sys.last_type = sys.last_value = sys.last_traceback = None
gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.is_available(), "No GPU: Kaggle -> Accelerator GPU T4 x2; Colab -> Runtime -> Change runtime type"
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.mem_get_info(0)[0] / 2**30:.1f} GB free; overflow layers spill to CPU)")

# the same load as 01b and the N-ATLaS model card: plain float16, no quantisation.
# transformers >= 4.56 calls it `dtype`; older versions use `torch_dtype`.
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
from transformers import AutoModelForCausalLM, AutoTokenizer
load_kw = {DTYPE_KW: torch.float16, "device_map": "auto", "token": HF_TOKEN, "low_cpu_mem_usage": True}
print("loading", BASE_MODEL, "in float16 ...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kw)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model.eval()
model.generation_config.max_length = None           # avoids the "max_new_tokens and max_length" warning
tok.padding_side = "left"
if tok.pad_token is None or tok.pad_token == tok.eos_token:
    tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() else tok.eos_token
INPUT_DEVICE = model.get_input_embeddings().weight.device

def chat(user_msgs, system=None, max_new=MAX_NEW):
    texts = []
    for u in user_msgs:
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": u}]
        texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(INPUT_DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

print(chat([T_PROMPT.format(lang="Hausa", extra="", text="The child has fever for 3 days.")], max_new=60)[0])

## 5. Translate (resumable)
Each job is one string: `(English id, language, field)` with field = question, steps or danger.
Finished strings are stored in `translations_raw_{STAGE}.jsonl`; a re-run skips them.

In [ ]:
def _retry_upload(api, progress, **kw):
    # Hugging Face can answer 429/500 when several jobs commit at once: retry with backoff;
    # a failed PROGRESS save only warns (the run continues and saves again later)
    import time
    for i in range(6):
        try:
            return api.upload_file(**kw)
        except Exception as e:
            if i == 5:
                if progress:
                    print(f"WARNING: progress upload failed ({type(e).__name__}); continuing"); return None
                raise
            print(f"upload failed ({type(e).__name__}): retry {i + 1}/5 in {15 * 2 ** i}s", flush=True)
            time.sleep(15 * 2 ** i)

def upload(files, msg, progress=False):
    if not UPLOAD:
        return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in files:
        _retry_upload(api, progress, path_or_fileobj=f, path_in_repo=f"translations/{f}", repo_id=DATA_REPO,
                      repo_type="dataset", commit_message=msg)

parts = {r["id"]: split_answer(r["answer"]) for r in picked}
jobs = []
for r in picked:
    for lang in LANGS:
        for field, text in (("question", r["question"]), ("steps", parts[r["id"]]["steps"]),
                            ("danger", parts[r["id"]]["danger"])):
            jobs.append((r["id"], lang, field, text))
jobs.sort(key=lambda j: len(j[3]))                     # similar lengths per batch = faster

RAW = f"translations_raw_{STAGE}.jsonl"
done = {}
if not os.path.exists(RAW) and UPLOAD:
    try:
        import shutil
        from huggingface_hub import hf_hub_download
        shutil.copy(hf_hub_download(DATA_REPO, f"translations/{RAW}", repo_type="dataset", token=HF_TOKEN), RAW)
        print("Resuming from Hugging Face copy of", RAW)
    except Exception:
        pass
if os.path.exists(RAW):
    for l in open(RAW, encoding="utf-8"):
        x = json.loads(l)
        done[(x["en_id"], x["lang"], x["field"])] = x["text"]
todo = [j for j in jobs if (j[0], j[1], j[2]) not in done]
print(f"{len(jobs)} strings | {len(done)} done | {len(todo)} to translate")

t0, nb = time.time(), (len(todo) + BATCH - 1) // BATCH
with open(RAW, "a", encoding="utf-8") as f:
    for b in range(nb):
        batch = todo[b * BATCH:(b + 1) * BATCH]
        try:
            outs = chat([T_PROMPT.format(lang=LANG_NAME[l], extra=EXTRA[l], text=t) for _, l, _, t in batch])
        except torch.cuda.OutOfMemoryError:           # long text did not fit: retry one by one
            torch.cuda.empty_cache()
            print(f"  OOM in batch {b + 1}: retrying {len(batch)} items individually", flush=True)
            outs = [chat([T_PROMPT.format(lang=LANG_NAME[l], extra=EXTRA[l], text=t)])[0] for _, l, _, t in batch]
        for (eid, l, field, _), o in zip(batch, outs):
            done[(eid, l, field)] = clean_output(o)
            f.write(json.dumps({"en_id": eid, "lang": l, "field": field, "text": done[(eid, l, field)]},
                               ensure_ascii=False) + "\n")
        f.flush()
        el = time.time() - t0
        if (b + 1) % 5 == 0 or b + 1 == nb:
            print(f"batch {b + 1}/{nb}  {el / 60:.1f} min  ETA {el / (b + 1) * (nb - b - 1) / 60:.0f} min", flush=True)
        if (b + 1) % SAVE_EVERY == 0:
            upload([RAW], f"03_translate {STAGE}: progress {b + 1}/{nb}", progress=True)
upload([RAW], f"03_translate {STAGE}: finished")

## 6. Automatic quality flags
| Flag | Meaning |
|---|---|
| `numbers` | a number/dose in the English text is missing from the translation |
| `lang` | GlotLID thinks the text is not Hausa/Yoruba |
| `length` | translation is much shorter or longer than the English |
| `repeat` | the model looped (same phrase repeated) |
| `copied` | output is (almost) the English text |
| `empty` | nothing came back |
| `no_tones` | Yoruba text with no tone marks at all |

Note: `fasttext.predict()` crashes with NumPy 2 (Kaggle's default), so the low-level `f.predict` is used.

In [ ]:
lid = None
if USE_LID:
    try:
        import fasttext
        from huggingface_hub import hf_hub_download
        lid = fasttext.load_model(hf_hub_download("cis-lmu/glotlid", "model.bin"))
    except Exception as e:
        print("GlotLID unavailable, skipping language check:", e)
LID_LABEL = {"ha": "__label__hau_Latn", "yo": "__label__yor_Latn", "ig": "__label__ibo_Latn"}
NUM = re.compile(r"\d+(?:[.,]\d+)?")
YO_TONES = re.compile(r"[àáèéìíòóùúẹọṣÀÁÈÉÌÍÒÓÙÚẸỌṢ\u0300\u0301]")

def flags_for(src, out, lang):
    f = []
    if not out.strip():
        return ["empty"]
    if set(NUM.findall(src)) - set(NUM.findall(out)):
        f.append("numbers")
    if len(src) > 25 and not (0.5 <= len(out) / len(src) <= 3.0):
        f.append("length")
    words = out.lower().split()
    grams = collections.Counter(tuple(words[i:i + 4]) for i in range(len(words) - 3))
    if grams and grams.most_common(1)[0][1] > 3:
        f.append("repeat")
    if src.strip().lower() == out.strip().lower() or (len(src) > 25 and src.lower()[:25] in out.lower()):
        f.append("copied")
    if lid is not None and len(out.split()) >= 4:
        label = lid.f.predict(out.replace("\n", " "), 1, 0.0, "strict")[0][1]
        if label != LID_LABEL[lang]:
            f.append("lang")
    if lang == "yo" and len(out.split()) >= 4 and not YO_TONES.search(out):
        f.append("no_tones")
    return f

assert flags_for("Give 250 mg twice daily", "Ba da mg sau biyu a rana", "ha") [:1] == ["numbers"]
assert "repeat" in flags_for("x " * 10, "a b c d a b c d a b c d a b c d a b c d", "ha")
print("flag self-check passed")

## 7. Build the review sheet
* `key_facts` are copied from English, but only facts that also appear in the translated answer are
  kept automatically (numbers and drug names usually survive translation; English words do not).
  `key_facts_en` keeps the original list so the reviewer can add the right Hausa/Yoruba words.
* `must_review` = yes for every benchmark row, and for training rows that are flagged or in a random
  1-in-5 sample.
* Flagged rows are sorted to the top.

In [ ]:
rng = random.Random(11)
out_rows = []
for r in picked:
    p = parts[r["id"]]
    for lang in LANGS:
        q = done.get((r["id"], lang, "question"), "")
        steps = done.get((r["id"], lang, "steps"), "")
        danger = done.get((r["id"], lang, "danger"), "")
        ans = build_answer({**p, "steps": steps, "danger": danger})
        fl = flags_for(r["question"], q, lang) + flags_for(p["steps"], steps, lang) \
             + ([] if p["danger"].lower() == "none listed" else flags_for(p["danger"], danger, lang))
        _sm = re.search(r"SOURCE:\s*(.*)", ans)
        if (_sm.group(1).strip().lower() if _sm else "") != (r.get("source") or "").strip().lower():
            fl.append("source_mismatch")               # SOURCE line in the built answer != the row's gold source
        fl = sorted(set(fl))
        facts_en = [k.strip() for k in r["key_facts"].split(";") if k.strip()]
        facts = [k for k in facts_en if k.lower() in ans.lower()]
        must = STAGE == "bench" or bool(fl) or rng.random() < 0.2
        out_rows.append({
            "id": r["id"].replace("EN-", lang.upper() + "-", 1), "en_id": r["id"], "lang": lang,
            "split": r["split"], "kind": r["kind"], "chunk_id": r["chunk_id"], "source": r["source"],
            "action": p["action"], "flags": ";".join(fl), "must_review": "yes" if must else "no",
            "question_en": r["question"], "question": q,
            "answer_en": r["answer"], "answer": ans,
            "key_facts_en": "; ".join(facts_en), "key_facts": "; ".join(facts),
            "status": "", "reviewer": "", "comment": ""})

out_rows.sort(key=lambda x: (x["lang"], x["flags"] == "", x["must_review"] == "no", x["id"]))
OUT = f"review_ha_yo_{STAGE}.csv"
with open(OUT, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(out_rows[0]))
    w.writeheader(); w.writerows(out_rows)
upload([OUT], f"03_translate {STAGE}: review sheet")

for lang in LANGS:
    L = [x for x in out_rows if x["lang"] == lang]
    fc = collections.Counter(fl for x in L for fl in x["flags"].split(";") if fl)
    print(f"{LANG_NAME[lang]}: {len(L)} rows | flagged {sum(bool(x['flags']) for x in L)} | "
          f"must_review {sum(x['must_review'] == 'yes' for x in L)} | flags {dict(fc)}")
ex = out_rows[0]
print("\nExample (", ex["id"], ex["flags"] or "no flags", ")\nEN:", ex["question_en"], "\n-> ", ex["question"], "\n" + ex["answer"])

## Reviewer instructions (paste into the sheet's first tab)
1. Filter `must_review = yes`. Flagged rows are on top.
2. Compare `question`/`answer` with the English columns. Fix the translation in place.
3. **Never change** `ACTION:`, the four labels or the `SOURCE:` line. Never change numbers or doses.
4. Yoruba: add correct tone marks and under-dots.
5. Benchmark rows: make `key_facts` 2–4 facts separated by `;` that appear **in the translated
   answer** (drug names and numbers are best; add the Hausa/Yoruba word where needed).
6. Set `status` = `ok`, `fixed` or `drop`, and your initials in `reviewer`.

When done, download as CSV and upload to Hugging Face as `reviewed/review_ha_yo_final.csv`
(bench and train rows together in one file; Part 6 reads it).

## Done when
* **Sun 4 Oct:** `translations/review_ha_yo_bench.csv` exists and is with the native speakers; reviewed
  version uploaded before the Mon 5 Oct freeze.
* **Mon 5 – Tue 6 Oct:** `translations/review_ha_yo_train.csv` reviewed (flagged + 1 in 5). If more than
  30% of the sampled rows of a language are bad, keep only that language's reviewed rows for training.